In [1]:
# v8: 2-step training — Step1 pretrain on full 83K rows, Step2 finetune on clean 4.8K rows
# Motivation: clean-data-only (v7) severely overfits due to small dataset size
# Architecture: same as v4 (bipolar montage, KL loss, dropout)
import torch
import os
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
import huggingface_hub

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt 

from timm import create_model  # or torchvision.models
from tqdm.notebook import tqdm
from torchinfo import summary

from torch.utils.data import DataLoader, random_split

# Step 2d: two-step training with Step1 also weighted

Step 2d: two-step training with Step1 also using WeightedRandomSampler on the
full 83K dataset (aligned with HMS 2nd-place / Bhatti et al. approach of
combining vote-weighting with staged training, rather than treating
reweighting and two-step as mutually exclusive).

Compare against v8 (Step 2c, pure two-step, no Step1 weighting).

Derived from `cnn-1d-v8-two-step.ipynb` (Step 2c) after its checkpoint-selection
bugfix (select on val_kl, not macro_f1). **The only difference from v8 is that
Step1 uses a class-balanced `WeightedRandomSampler`** instead of `shuffle=True`;
Step2 is unchanged (same `STEP2_LR=1e-5`, not the `1e-4` stress-test variant —
this notebook isolates the "does Step1 need weighting" question alone).

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../project/src')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
sys.path.insert(0, os.path.abspath('../project') if not IS_KAGGLE else '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code')

import yaml
CONFIG_PATH = os.path.join(CODE_DIR, 'config.yaml') if IS_KAGGLE else '../project/config/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print(cfg)

{'seed': 42, 'deterministic': True, 'data': {'train_meta': 'data_meta_splits/train_test_split.csv', 'val_meta': 'data_meta_splits/train_test_split.csv', 'eeg_dir': 'data_eeg_preprocessed/eeg_npy', 'spec_dir': 'data_eeg_preprocessed/spec_npy'}, 'model': {'name': 'eeg_1d_small', 'num_classes': 6}, 'loss': {'name': 'ce', 'class_weight': [], 'label_smoothing': 0.0}, 'train': {'run_name': 'exp', 'out_dir': 'runs', 'batch_size': 64, 'num_workers': 8, 'epochs': 50, 'lr': 0.0015, 'weight_decay': 0.01, 'amp': True, 'grad_clip': 1.0, 'use_weighted_sampler': False, 'early_stop_patience': 10, 'early_stop_key': 'macro_f1', 'lr_mode': 'cos'}, 'log': {'use_wandb': False, 'project': 'EEG-HBA'}}


In [ ]:
# ========== ENVIRONMENT CONFIG ==========
if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    META_PATH = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_clean.csv'
else:
    DATA_ROOT = os.path.abspath('../')
    META_PATH = os.path.abspath('../kaggle_upload/train_clean.csv')

# ========== REPRODUCIBILITY ==========
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE}")

# v3: soft label + KL div loss + dropout (backbone=0.15, head=0.4)

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/')
from src.data import load_clean_data, load_full_data, EEGDatasetV2, get_sampler, preprocess_eeg_windows
from torch.utils.data import DataLoader

CLEAN_PATH  = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_clean.csv'
FULL_PATH   = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
PARQUET_DIR = os.path.join(DATA_ROOT, 'train_eegs')

# 2-step data
full_train_df  = load_full_data(FULL_PATH,  split='trainval')  # 83K rows, Step1
clean_train_df = load_clean_data(CLEAN_PATH, split='trainval') # 4.8K rows, Step2
val_df         = load_clean_data(CLEAN_PATH, split='test')     # 1.1K rows, shared

print(f'Step1 train: {len(full_train_df):,} rows')
print(f'Step2 train: {len(clean_train_df):,} rows')
print(f'Val/test   : {len(val_df):,} rows')

# add target col + normalize votes -> probabilities (required by EEGDatasetV2)
_VOTE_COLS = ['seizure_vote','lpd_vote','gpd_vote','lrda_vote','grda_vote','other_vote']
for _d in [full_train_df, clean_train_df, val_df]:
    _d['target'] = _d['label']
    _vs = _d[_VOTE_COLS].sum(axis=1)
    _d['soft_y'] = list(_d[_VOTE_COLS].div(_vs, axis=0).to_numpy(dtype=np.float32))


In [ ]:
# ========== NPY CACHE (I/O speedup — see kaggle_run/src/data.py:preprocess_eeg_windows) ==========
cfg['cache_dir'] = cfg.get('cache_dir', '/tmp/hms_eeg_cache')
CACHE_DIR = cfg['cache_dir']
os.makedirs(CACHE_DIR, exist_ok=True)

existing = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
if len(existing) == 0:
    print(f'Building EEG window cache → {CACHE_DIR}')
    preprocess_eeg_windows(pd.concat([full_train_df, clean_train_df, val_df]), PARQUET_DIR, CACHE_DIR)
    print(f'Cache built: {len(os.listdir(CACHE_DIR))} files')
else:
    print(f'Cache already exists: {len(existing)} files, skipping')

In [ ]:
# Batch shape verification (Step 1 — full data sample)
_preview_ds = EEGDatasetV2(full_train_df.head(16), parquet_dir=PARQUET_DIR, cache_dir=CACHE_DIR)
_preview_dl = DataLoader(_preview_ds, batch_size=4, shuffle=False, num_workers=0)
batch = next(iter(_preview_dl))
print('x shape:', batch['x'].shape)
print('y shape:', batch['y'].shape)
print('y sample:', batch['y'][:4])


In [6]:
from models.classifier import build_model

model = build_model(cfg["model"]).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {trainable:,}")

Parameters — total: 217,030 | trainable: 217,030


In [ ]:
# ========== SHARED EVALUATION SETUP ==========
# Identical to cnn-1d-v7-clean-data.ipynb / cnn-1d-v7b-weighted-sampler.ipynb
# so Step 2a / 2b / 2c val numbers are directly comparable.
# src/evaluation.validate() divides by len(loader.dataset) (not len(loader)),
# so an uneven last batch no longer biases val KL; soft_key="soft_y" scores
# against the true vote distribution rather than a one-hot of the hard label.
from src.evaluation import validate as shared_validate
from src.losses import build_loss

eval_cfg = {"loss": {"name": "kl", "label_smoothing": 0.0},
            "model": {"num_classes": 6}}
# NOTE: must be the KLLoss from build_loss() — it applies log_softmax itself.
# Passing a bare nn.KLDivLoss() here would skip that and give wrong numbers.
val_loss_fn = build_loss(eval_cfg)

In [ ]:
# v8c: 2-step training — Step1 also weighted (bipolar montage, KL loss, dropout)
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

GRAD_CLIP    = cfg['train']['grad_clip']
USE_AMP      = cfg['train']['amp'] and DEVICE.type == 'cuda'
WEIGHT_DECAY = cfg['train']['weight_decay']
PATIENCE     = cfg['train'].get('early_stop_patience', 10)
STEP2_PATIENCE = 15   # Step2 converges slowly, needs more patience
BATCH_SIZE   = cfg['train']['batch_size']

# 2-step hyperparameters — identical to v8 (Step 2c), only Step1 sampling differs
STEP1_EPOCHS = 20
STEP2_EPOCHS = 30
STEP1_LR     = 1e-3
STEP2_LR     = 1e-5   # smaller LR for finetune — same as v8, not the 1e-4 stress-test variant

criterion = nn.KLDivLoss(reduction='batchmean')   # training loss only
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

history = {'step1': [], 'step2': []}

# shared val_loader (clean test split, 1,139 rows)
val_ds     = EEGDatasetV2(val_df, parquet_dir=PARQUET_DIR, cache_dir=CACHE_DIR)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

# Validation is scored by shared_validate() (src/evaluation.py); the old inline
# validate() lived here and is gone. It also tracked val CE, which is dropped —
# the paper reports KL and macro F1 only.


def train_phase(phase_name, train_df_phase, epochs, lr, save_path, hist_key,
                 patience=None, use_sampler=False):
    # One training phase with early stopping. Mutates model in-place.
    # use_sampler=True: class-balanced WeightedRandomSampler instead of
    # shuffle=True — this is the only change from v8 (Step 2c), applied to
    # Step1 only.
    if patience is None:
        patience = PATIENCE
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_kl, best_epoch, best_f1_at_best_kl, wait = float('inf'), 0, 0.0, 0
    best_state = None

    phase_ds = EEGDatasetV2(train_df_phase, parquet_dir=PARQUET_DIR, cache_dir=CACHE_DIR)
    if use_sampler:
        sampler = get_sampler(train_df_phase, label_col='label')
        phase_loader = DataLoader(phase_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=2, pin_memory=True)
    else:
        phase_loader = DataLoader(phase_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=True)

    for epoch in range(1, epochs + 1):
        # train
        model.train()
        t0 = time.time()
        train_loss = 0.0
        for batch in phase_loader:
            x      = batch['x'].to(DEVICE)
            soft_y = batch['soft_y'].to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits    = model(x)
                loss      = criterion(F.log_softmax(logits, dim=1), soft_y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        scheduler.step()

        # validate (shared: src/evaluation.validate)
        val_loss, metrics, val_probs, val_y_true = shared_validate(
            model, val_loader, val_loss_fn, eval_cfg, DEVICE, soft_key='soft_y'
        )
        val_kl    = val_loss
        macro_f1  = metrics['macro_f1']
        avg_train = train_loss / len(phase_loader)
        elapsed   = time.time() - t0

        history[hist_key].append({
            'epoch': epoch, 'train_kl': avg_train,
            'val_kl': val_kl, 'macro_f1': macro_f1,
        })
        print(f'  Epoch {epoch:03d} | train_kl {avg_train:.4f} | '
              f'val_kl {val_kl:.4f} | '
              f'macro_f1 {macro_f1:.4f} | {elapsed:.0f}s')

        # checkpoint / early-stop on val_kl (the headline metric), not macro_f1
        if val_kl < best_kl:
            best_kl, best_epoch, best_f1_at_best_kl, wait = val_kl, epoch, macro_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), save_path)
            print(f'    ✓ saved {save_path} (kl={best_kl:.4f})')
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'  {phase_name} done — best epoch {best_epoch}: '
          f'val_kl={best_kl:.4f}, macro_f1={best_f1_at_best_kl:.4f}')
    if best_state is not None:
        model.load_state_dict(best_state)


# Step 1: pretrain on full data — WeightedRandomSampler (the change under test)
print('=' * 50)
print(f'Step 1: pretrain on full data ({len(full_train_df):,} rows), weighted sampler')
print('=' * 50)
train_phase('Step 1', full_train_df, STEP1_EPOCHS, STEP1_LR,
            'best_v8c_step1.pt', 'step1', use_sampler=True)

# Load Step1 best checkpoint before Step2 fine-tune
model.load_state_dict(torch.load('best_v8c_step1.pt'))
print('Loaded Step1 best checkpoint')

# Step 2: finetune on clean data (unchanged from v8 — no sampler)
print('\n' + '=' * 50)
print(f'Step 2: finetune on clean data ({len(clean_train_df):,} rows)')
print('=' * 50)
train_phase('Step 2', clean_train_df, STEP2_EPOCHS, STEP2_LR,
            'best_v8c_step2.pt', 'step2', patience=STEP2_PATIENCE)

print(f'\n2-step training complete.')
for _key, _label in [('step1', 'Step1'), ('step2', 'Step2')]:
    _best = min(history[_key], key=lambda h: h['val_kl'])
    print(f'  {_label} best: val_kl={_best["val_kl"]:.4f}, '
          f'macro_f1={_best["macro_f1"]:.4f} (epoch {_best["epoch"]})')

In [ ]:
import matplotlib.pyplot as plt

# val CE panel dropped — shared_validate() reports KL and macro F1 only
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (phase_key, phase_label) in enumerate([('step1', 'Step1 (full data, weighted)'), ('step2', 'Step2 (clean finetune)')]):
    hist = history[phase_key]
    if not hist:
        continue
    epochs_   = [h['epoch']    for h in hist]
    train_kl_ = [h['train_kl'] for h in hist]
    val_kl_   = [h['val_kl']   for h in hist]
    macro_f1_ = [h['macro_f1'] for h in hist]

    ax1, ax2 = axes[row]
    ax1.plot(epochs_, train_kl_, label='train KL')
    ax1.plot(epochs_, val_kl_,   label='val KL')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('KL Divergence')
    ax1.set_title(f'KL Divergence — {phase_label}'); ax1.legend()

    ax2.plot(epochs_, macro_f1_, color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
    ax2.set_title(f'Val Macro F1 — {phase_label}')

plt.tight_layout()
plt.savefig('training_curves_v8c.png', dpi=150)
plt.show()

# Report the best-val_kl epoch's own macro_f1 (the checkpointed model),
# not an independently-best macro_f1 from a different epoch.
for phase_key, phase_label in [('step1', 'Step1'), ('step2', 'Step2')]:
    hist = history[phase_key]
    if hist:
        best = min(hist, key=lambda h: h['val_kl'])
        print(f'{phase_label} best: val_kl={best["val_kl"]:.4f}, '
              f'macro_f1={best["macro_f1"]:.4f} (epoch {best["epoch"]})')

In [ ]:
# One-line summary per phase — copy into notebooks/RESULTS.md
# Both numbers come from the best-val_kl epoch of that phase (the checkpoint
# that was actually saved), so val_kl and val_f1 are from the same model.
for _key, _label in [('step1', 'cnn-1d-v8c-weighted-two-step (Step1 pretrain, full data, weighted)'),
                     ('step2', 'cnn-1d-v8c-weighted-two-step (Step2 finetune, clean data)')]:
    _h = history[_key]
    if _h:
        _best = min(_h, key=lambda h: h['val_kl'])
        print(f"{_label} | "
              f"val_kl={_best['val_kl']:.4f} | "
              f"val_f1={_best['macro_f1']:.4f} | "
              f"epoch={_best['epoch']}")